We previously explored GLM via Lasso+Gaussian with log10, and achieve 0.07 R^2, We might wanna explore Beta regression with lasso penalty, which is more suitable for our data. We will use the same train/test split as before, and use the same set of covariates.

In [62]:
# load lib
library(tidyverse)
library(glmnet)
library(glmmTMB)
library(caret)
library(DHARMa)
library(readr)
library(dplyr)
library(janitor)

In [63]:
df <- read_csv("data/merged_with_svi.csv") |> clean_names()
df <- df %>% filter(!county %in% c("Mcculloch", "Mclennan", "Mcmullen", "Dewitt", "Loving"))
df_raw <- df
county <- df$county
df$density <- df$population * 1.0 / df$area_sqmi

# Keep enrollment, outbreak, phr for now — only drop the feature columns
df <- df %>% select(
  -c("county", "area_sqmi"),
  -starts_with("m_"), -starts_with("mp_"),
  -starts_with("e_"), -starts_with("epl_"),
  -starts_with("spl_"), -starts_with("rpl_"),
  -starts_with("f_")
)

df <- df %>% select(-c("ep_minrty", "ep_hisp", "ep_afam", "ep_pov150", "ep_uninsur"))
df <- df[, colSums(is.na(df)) == 0]

set.seed(100)
perc_strata <- 0.75
strata <- ifelse(df$outbreak > 0, "nonzero", "zero")
index <- createDataPartition(strata, p = perc_strata, list = FALSE)
train <- df[index, ]
test  <- df[-index, ]

outcome <- train$outbreak
offset  <- log(train$enrollment)
phr     <- train$phr


Rows: 254 Columns: 556
-- Column specification --------------------------------------------------------
Delimiter: ","
chr   (1): County
dbl (532): cve, outbreak, enrollment, population, PHR, pct_hispanic, pct_bla...
lgl  (23): median_income, Advised to Cut Down Salt - Do not use salt, Diabet...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [64]:
X <- as.matrix(train)
y <- outcome
X <- X[, abs(cor(X, log(y + 1), use = "complete.obs", method = "spearman")) >= 0.1]

# Start with full list of candidates
candidates <- colnames(X)

i <- 1
while (i < length(candidates)) {
  j <- i + 1
  while (j <= length(candidates)) {
    rho <- cor(X[, candidates[i]], X[, candidates[j]], 
               method = "spearman", use = "complete.obs")
    if (abs(rho) >= 0.7) {
      # Drop whichever has weaker correlation with outcome
      rho_i <- abs(cor(X[, candidates[i]], log(y + 1), method = "spearman", use = "complete.obs"))
      rho_j <- abs(cor(X[, candidates[j]], log(y + 1), method = "spearman", use = "complete.obs"))
      if (rho_i >= rho_j) {
        candidates <- candidates[-j]  
      } else {
        candidates <- candidates[-i]  
        j <- i + 1
      }
    } else {
      j <- j + 1
    }
  }
  i <- i + 1
}

X <- X[, candidates]
X

cve,outbreak,pct_hispanic,pct_uninsured,bmi_3_categories_recommended_range,clnscpy_sgmscpy_colonoscopy,ever_had_hiv_test_no,general_health_good,heavy_drinking_no,hysterectomy_no,...,up_to_date_crc_scrn_age_50_7_no,ep_nohsdp,ep_age65,ep_age17,ep_disabl,ep_sngpnt,ep_munit,ep_mobile,ep_aian,ep_nhpi
2.54,0,19.6,18.5,25.4,83.3,62.4,30.9,94.9,71.4,...,27.7,15.2,14.8,19.0,16.8,7.8,3.5,19.7,0.7,0.1
2.50,0,23.5,17.7,23.8,75.5,55.7,33.1,93.1,64.6,...,37.6,16.0,16.3,25.4,19.1,9.4,6.1,18.2,0.4,0.0
5.24,0,12.0,4.2,21.7,87.6,63.5,35.0,95.9,81.1,...,32.4,7.2,28.7,22.2,11.7,4.8,0.0,7.7,1.1,0.0
1.08,1,65.4,19.5,28.7,84.6,59.1,33.9,91.0,80.0,...,25.9,19.4,14.8,26.9,11.8,7.9,1.4,33.8,0.0,0.0
0.80,2,66.0,28.6,21.7,87.6,63.5,35.0,95.9,81.1,...,32.4,26.8,12.4,26.9,13.0,12.4,0.2,9.4,1.0,0.0
3.78,0,21.0,13.3,28.7,84.6,59.1,33.9,91.0,80.0,...,25.9,8.0,27.9,16.8,18.8,3.4,0.2,26.0,0.5,0.0
2.09,0,45.0,21.8,34.1,86.3,60.4,34.7,92.8,83.0,...,22.7,16.9,15.4,25.5,12.1,4.9,2.2,24.4,0.1,0.0
1.48,0,61.8,18.8,27.0,77.9,60.4,31.1,97.3,74.2,...,39.2,19.8,12.6,20.2,15.2,6.5,3.0,15.7,0.3,0.0
2.42,0,26.3,14.0,34.1,86.3,60.4,34.7,92.8,83.0,...,22.7,9.2,11.3,27.5,14.8,10.5,10.1,6.4,0.2,0.6
2.01,1,59.7,16.0,28.7,84.6,59.1,33.9,91.0,80.0,...,25.9,14.1,12.3,25.1,14.3,8.4,17.6,2.7,0.1,0.1


In [71]:
library(bamlss)
set.seed(100)
fam <- beta_bamlss()
print(fam$names)

enrollment <- exp(offset)
rate <- y / enrollment

n <- length(rate)
rate_beta <- (rate * (n - 1) + 0.5) / n

dat <- data.frame(
  outbreak_rate = rate_beta,
  enrollment = enrollment,
  PHR = phr,
  X
)

xvars <- colnames(X)

la_terms <- paste0("la(", xvars, ")", collapse = " + ")

f <- as.formula(
  paste("outbreak_rate ~", la_terms)
)

fit <- bamlss(
  formula = f,
  family = fam,
  data = dat,
  sampler = FALSE,
  optimizer = opt_lasso,
  criterion = "BIC",
  nlambda = 100
)

[1] "mu"     "sigma2"


Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting 

BIC -1867.33 edf 34.364 lambda 1000.0 iteration   1

Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting 


BIC -1867.61 edf 34.414 lambda 869.74 iteration   2

...


BIC -1878.58 edf 36.000 lambda 0.0013 iteration  98

Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting 


BIC -1878.69 edf 36.000 lambda 0.0011 iteration  99

Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting 


BIC -1878.79 edf 36.000 lambda 0.0010 iteration 100
elapsed time: 33.54sec


In [72]:
mstop <- lasso_stop(fit)

coef_fit <- coef(fit, mstop = mstop)
print(coef_fit)

                                                                              mu.s.la(cve).cve 
                                                                                  9.003191e-03 
                                                                            mu.s.la(cve).tau21 
                                                                                  1.000000e+03 
                                                                    mu.s.la(outbreak).outbreak 
                                                                                  7.626335e-03 
                                                                       mu.s.la(outbreak).tau21 
                                                                                  1.000000e+03 
                                                            mu.s.la(pct_hispanic).pct_hispanic 
                                                                                 -2.913360e-03 
                                        

In [73]:
b <- unlist(coef_fit)

# keep only actual covariate coefficients from la() terms
selected <- b[
  grepl("^mu\\.s\\.la\\(", names(b)) &
    !grepl("\\.tau", names(b)) &
    abs(b) > 1e-5
]

selected_vars <- names(selected)

# extract variable names inside la(...)
selected_vars <- sub("^mu\\.s\\.la\\(([^\\)]+)\\).*", "\\1", selected_vars)

selected_table <- data.frame(
  variable = selected_vars,
  coefficient = as.numeric(selected),
  row.names = NULL
)

selected_table

selected_table <- selected_table[
  order(abs(selected_table$coefficient), decreasing = TRUE),
]

selected_table

variable,coefficient
<chr>,<dbl>
cve,0.009003191
outbreak,0.007626335
pct_hispanic,-0.002913360
pct_uninsured,0.008002541
bmi_3_categories_recommended_range,-0.102415383
clnscpy_sgmscpy_colonoscopy,-0.033289034
ever_had_hiv_test_no,-0.011718654
general_health_good,0.029325851
heavy_drinking_no,-0.011278389


,variable,coefficient
,<chr>,<dbl>
34,ep_nhpi,-0.130902981
5,bmi_3_categories_recommended_range,-0.102415383
12,last_dentist_visit_within_the_past_2_years,-0.062247057
25,up_to_date_crc_scrn_age_50_7_no,-0.044612831
33,ep_aian,0.041701942
22,smoker_status_former_smoker,-0.034001708
6,clnscpy_sgmscpy_colonoscopy,-0.033289034
8,general_health_good,0.029325851
18,remove_teeth_1_to_5,-0.025848226


In [68]:
selected_vars <- selected_table$variable[
  abs(selected_table$coefficient) >= 0.03
]

In [69]:
library(betareg)
train$outbreak_rate <- train$outbreak / train$enrollment
test$outbreak_rate  <- test$outbreak / test$enrollment
n_train <- nrow(train)
n_test  <- nrow(test)

train$outbreak_rate <- (train$outbreak_rate * (n_train - 1) + 0.5) / n_train
test$outbreak_rate  <- (test$outbreak_rate  * (n_test - 1) + 0.5) / n_test

train_df_small <- train[, c("outbreak_rate", selected_vars), drop = FALSE]
test_df_small  <- test[,  c("outbreak_rate", selected_vars), drop = FALSE]

form_beta <- as.formula(
  paste("outbreak_rate ~", paste(selected_vars, collapse = " + "))
)

beta_fit <- betareg(
  form_beta,
  data = train_df_small
)

summary(beta_fit)


Call:
betareg(formula = form_beta, data = train_df_small)

Quantile residuals:
    Min      1Q  Median      3Q     Max 
-0.5958 -0.2281 -0.1542 -0.0302     Inf 

Coefficients (mean model with logit link):
                                             Estimate Std. Error z value
(Intercept)                                -7.1133883  2.2669309  -3.138
ep_nhpi                                    -0.0495991  0.2685074  -0.185
bmi_3_categories_recommended_range         -0.0197304  0.0255704  -0.772
last_dentist_visit_within_the_past_2_years  0.0112333  0.0211336   0.532
up_to_date_crc_scrn_age_50_7_no            -0.0001608  0.0142751  -0.011
ep_aian                                    -0.0160993  0.0836495  -0.192
smoker_status_former_smoker                -0.0038368  0.0258031  -0.149
clnscpy_sgmscpy_colonoscopy                 0.0233762  0.0152227   1.536
                                           Pr(>|z|)   
(Intercept)                                  0.0017 **
ep_nhpi                    

In [60]:
selected_vars2 <- c("ep_aian", "cve", "ep_nhpi", "bmi_3_categories_recommended_range", "last_dentist_visit_within_the_past_2_years", "up_to_date_crc_scrn_age_50_7_no", "clnscpy_sgmscpy_colonoscopy", "ep_age17", "ep_age65", "pct_uninsured")

train_df_small2 <- train[, c("outbreak_rate", selected_vars2), drop = FALSE]
test_df_small2  <- test[,  c("outbreak_rate", selected_vars2), drop = FALSE]

form_beta2 <- as.formula(
  paste("outbreak_rate ~", paste(selected_vars2, collapse = " + "))
)

beta_fit2 <- betareg(
  form_beta2,
  data = train_df_small2
)

summary(beta_fit2)


Call:
betareg(formula = form_beta2, data = train_df_small2)

Quantile residuals:
    Min      1Q  Median      3Q     Max 
-4.0057 -0.4326  0.0136  0.5352  7.0986 

Coefficients (mean model with logit link):
                                            Estimate Std. Error z value
(Intercept)                                -6.103112   1.458453  -4.185
ep_aian                                     0.127646   0.058676   2.175
cve                                         0.169402   0.009715  17.437
ep_nhpi                                    -0.100327   0.201686  -0.497
bmi_3_categories_recommended_range         -0.022337   0.011842  -1.886
last_dentist_visit_within_the_past_2_years -0.014158   0.012530  -1.130
up_to_date_crc_scrn_age_50_7_no            -0.018639   0.010362  -1.799
clnscpy_sgmscpy_colonoscopy                 0.004618   0.011587   0.399
ep_age17                                    0.019475   0.010475   1.859
ep_age65                                   -0.029772   0.007200  -4.135


In [61]:
pred_test <- predict(beta_fit2, newdata = test_df_small2, type = "response")

test_y <- test_df_small2$outbreak_rate

rmse <- sqrt(mean((test_y - pred_test)^2))
mae  <- mean(abs(test_y - pred_test))

sse <- sum((test_y - pred_test)^2)
sst <- sum((test_y - mean(test_y))^2)
r2_test <- 1 - sse / sst

c(RMSE = rmse, MAE = mae, R2_test = r2_test)

RMSE          MAE      R2_test 
 0.006113074  0.005667974 -4.449198565

In [26]:
pred_test1 <- predict(beta_fit,  newdata = test_df_small,  type = "response")
pred_test2 <- predict(beta_fit2, newdata = test_df_small2, type = "response")

eval <- function(y, pred) {
  c(
    RMSE = sqrt(mean((y - pred)^2)),
    MAE = mean(abs(y - pred)),
    R2_test = 1 - sum((y - pred)^2) / sum((y - mean(y))^2)
  )
}

rbind(
  model_14_vars = eval(test_df_small$outbreak_rate, pred_test1),
  model_9_vars  = eval(test_df_small2$outbreak_rate, pred_test2)
)

,RMSE,MAE,R2_test
model_14_vars,0.006181510,0.005719509,-4.571889
model_9_vars,0.006200048,0.005703789,-4.605359


In [70]:
eval <- function(fit, y_true, pred) {
  spearman  <- cor(pred, y_true, method = "spearman")
  pseudo_r2 <- summary(fit)$pseudo.r.squared   # log-likelihood based, train
  sse <- sum((y_true - pred)^2)
  sst <- sum((y_true - mean(y_true))^2)
  c(
    Pseudo_R2_train = pseudo_r2,
    Spearman_rho    = spearman,
    RMSE            = sqrt(mean((y_true - pred)^2)),
    MAE             = mean(abs(y_true - pred))
  )
}

rbind(
  model_14_vars = eval(beta_fit,  test_df_small$outbreak_rate,  pred_test1),
  model_9_vars  = eval(beta_fit2, test_df_small2$outbreak_rate, pred_test2)
)

,Pseudo_R2_train,Spearman_rho,RMSE,MAE
model_14_vars,0.1154509,0.2941579,0.006181510,0.005719509
model_9_vars,0.3544667,0.2728677,0.006200048,0.005703789


In [27]:
test_y <- test_df_small2$outbreak_rate

baseline_pred <- rep(mean(test_y), length(test_y))

baseline_rmse <- sqrt(mean((test_y - baseline_pred)^2))
baseline_mae  <- mean(abs(test_y - baseline_pred))

c(
  model_RMSE = rmse,
  baseline_RMSE = baseline_rmse,
  model_MAE = mae,
  baseline_MAE = baseline_mae
)

model_RMSE baseline_RMSE     model_MAE  baseline_MAE 
 0.0062000476  0.0026187456  0.0057037887  0.0007889461